# E-commerce source assessment

A read-only, aggregate-only assessment of the seven source files. No customer names, raw rows, local paths, or generated datasets are displayed or written. Results describe the local snapshot and may change with the inputs. Run with Python 3 and pandas.

Source files are Git-ignored. Put a separately obtained copy in the repository’s local `datasets/E-Commerce Sales Dataset` folder or set `ECOMMERCE_SOURCE_DIR` to the source folder. Open this notebook from the project or repository; run all cells. Saved output shows aggregate results for the inspected local snapshot; results may change when the source files change.


In VS Code, install the Python and Jupyter extensions and select a Python environment with `pandas` and `ipykernel` installed. You can install those packages in a project environment with `python -m pip install pandas ipykernel`. Then choose **Run All**. Do not save executed outputs for publication without reviewing them.

In [1]:
from pathlib import Path
from collections import Counter
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import csv, hashlib, os
import pandas as pd

SOURCES = {
    'amazon': 'Amazon Sale Report.csv', 'inventory': 'Sale Report.csv',
    'international': 'International sale Report.csv', 'price_may': 'May-2022.csv',
    'price_march': 'P  L March 2021.csv', 'expense': 'Expense IIGF.csv',
    'warehouse': 'Cloud Warehouse Compersion Chart.csv',
}

def locate_sources():
    override = os.environ.get('ECOMMERCE_SOURCE_DIR')
    candidates = [Path(override).expanduser()] if override else [p / 'datasets' / 'E-Commerce Sales Dataset' for p in (Path.cwd(), *Path.cwd().parents)]
    for candidate in candidates:
        if all((candidate / f).is_file() for f in SOURCES.values()):
            return candidate
    raise FileNotFoundError('Seven source files not found. Set ECOMMERCE_SOURCE_DIR to their local folder.')

source_dir = locate_sources()
print('Seven source files found; local path and raw records are not displayed.')

Seven source files found; local path and raw records are not displayed.


## 1. Inventory and fingerprint

Count rows and columns and calculate SHA-256 for each local file. Hashes identify file contents; they do not establish ownership or permission to redistribute. Read all fields as text to preserve the original values.

In [2]:
frames, inventory = {}, []
for source, filename in SOURCES.items():
    path = source_dir / filename
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    with path.open(encoding='utf-8-sig', newline='') as stream:
        header = next(csv.reader(stream))
    if len(set(header)) != len(header):
        raise ValueError(f'Duplicate column names: {source}')
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig', on_bad_lines='error')
    frames[source] = df
    inventory.append({'source': source, 'rows': len(df), 'columns': len(df.columns), 'sha256': digest.hexdigest()})
print(pd.DataFrame(inventory).to_string(index=False))

       source   rows  columns                                                           sha256
       amazon 128975       24 ac9a366da5f3af418831fa53da9eebfc3b3838803e3774eec97dbcb291441beb
    inventory   9271        7 60320c9831d0194c52ce5791f03f9c4f12e3474490a931d7cd23282290a28e6d
international  37432       10 b0497693c317f456311f406a3a7d1cb15d38949143a50b6b15ab5c50c6e2baaf
    price_may   1330       17 f24628666e406e96515a5beeb8cfc6e4f2350594217800b67f3613e0d98c32a7
  price_march   1330       18 978679bb9d41e4b8a96dc5c65b0e0bae4b65546fa8a2f5f60457a93491811298
      expense     17        5 b1fa321a3aa963d91d06fc032a83848baec491cf13f9abd3ffbe75c6770c70db
    warehouse     50        4 3c8e296064b7f3b98021f573c9e6777673234e67a9673ddc248995fb9fbb4c90


## 2. Sales completeness and grain

Count missing measures and zero-quantity records, and inspect repeated order/SKU combinations. Repeated combinations alone do not prove duplicate transactions.

In [3]:
sales = frames['amazon']
blank = lambda s: s.str.strip().eq('')
norm = lambda s: s.str.strip().str.upper()
num = lambda s: pd.to_numeric(s.str.replace(',', '', regex=False).str.strip(), errors='coerce')
dates = pd.to_datetime(sales['Date'].str.strip(), format='%m-%d-%y', errors='coerce')
qty, amount = num(sales['Qty']), num(sales['Amount'])
missing_amount, missing_currency = blank(sales['Amount']), blank(sales['currency'])
pairs = sales.groupby(['Order ID', 'SKU'], dropna=False).size()
print('Distinct sale dates:', dates.nunique())
print('Date range:', dates.min().date(), 'through', dates.max().date())
print('Missing amount:', int(missing_amount.sum()))
print('Missing currency:', int(missing_currency.sum()))
print('Missing either amount or currency:', int((missing_amount | missing_currency).sum()))
print('Zero-quantity rows:', int(qty.eq(0).sum()))
print('Zero quantity with positive amount:', int((qty.eq(0) & amount.gt(0)).sum()))
print('Order/SKU pairs appearing more than once:', int(pairs.gt(1).sum()))

Distinct sale dates: 91
Date range: 2022-03-31 through 2022-06-29
Missing amount: 7795
Missing currency: 7795
Missing either amount or currency: 7795
Zero-quantity rows: 12807
Zero quantity with positive amount: 5136
Order/SKU pairs appearing more than once: 7


## 3. Inventory identity

Normalize SKU using trim and uppercase. Count all rows belonging to duplicate SKU groups, then apply the lakehouse stock validation to otherwise unambiguous rows.

In [4]:
inv = frames['inventory']
sku = norm(inv['SKU Code'])
counts = sku.value_counts()
missing = sku.eq('')
ambiguous = ~missing & sku.map(counts).gt(1)
stock = num(inv['Stock'])
invalid = ~missing & ~ambiguous & (stock.isna() | stock.abs().eq(float('inf')) | stock.lt(0))
print('Missing SKU rows:', int(missing.sum()))
print('Rows with ambiguous duplicate SKU:', int(ambiguous.sum()))
print('Other invalid stock rows:', int(invalid.sum()))
print('Accepted rows:', int((~(missing | ambiguous | invalid)).sum()))
print('Rejected rows:', int((missing | ambiguous | invalid).sum()))

Missing SKU rows: 83
Rows with ambiguous duplicate SKU: 23
Other invalid stock rows: 0
Accepted rows: 9165
Rejected rows: 106


## 4. International records

Check the source date, nonnegative quantity, and numeric rate and gross amount, including the lakehouse’s decimal precision rule. Missing SKU does not automatically reject a row.

In [5]:
def decimal_value(value, scale, precision):
    try:
        v = Decimal(value.strip().replace(',', ''))
        if not v.is_finite():
            return None
        rounded = v.quantize(Decimal(1).scaleb(-scale), rounding=ROUND_HALF_UP)
        return rounded if abs(rounded) < Decimal(10) ** (precision - scale) else None
    except (InvalidOperation, ValueError):
        return None

intl = frames['international']
valid_date = pd.to_datetime(intl['DATE'].str.strip(), format='%m-%d-%y', errors='coerce').notna()
quantity = intl['PCS'].map(lambda x: decimal_value(x, 4, 20))
rate2 = intl['RATE'].map(lambda x: decimal_value(x, 2, 20))
rate6 = intl['RATE'].map(lambda x: decimal_value(x, 6, 24))
gross2 = intl['GROSS AMT'].map(lambda x: decimal_value(x, 2, 20))
gross6 = intl['GROSS AMT'].map(lambda x: decimal_value(x, 6, 24))
valid = valid_date & quantity.map(lambda x: x is not None and x >= 0) & rate2.notna() & rate6.notna() & gross2.notna() & gross6.notna() & rate2.eq(rate6) & gross2.eq(gross6)
print('Accepted transaction-like rows:', int(valid.sum()))
print('Rejected rows:', int((~valid).sum()))
print('Accepted rows with missing SKU:', int((valid & blank(intl['SKU'])).sum()))

Accepted transaction-like rows: 18635
Rejected rows: 18797
Accepted rows with missing SKU: 1417


## 5. Cross-source SKU coverage

Exact normalized matches are not proof of shared ownership, timing, or causation. The first count uses inventory rows with nonblank, unambiguous SKUs; the second compares price-sheet rows with the sales SKU set.

In [6]:
sales_sku = norm(sales['SKU'])
unique_inventory = set(sku[~missing & ~ambiguous])
sales_set = set(sales_sku[sales_sku.ne('')])
print('Sales rows matching unambiguous inventory SKU:', int(sales_sku.isin(unique_inventory).sum()))
for source in ('price_may', 'price_march'):
    print(source, 'rows matching sales SKU:', int(norm(frames[source]['Sku']).isin(sales_set).sum()))

Sales rows matching unambiguous inventory SKU: 121271
price_may rows matching sales SKU: 0
price_march rows matching sales SKU: 0


## 6. Price observations

The lakehouse unpivots TP and channel price fields. Match the model’s blank, parsed, and unparsed classifications using its two- versus six-decimal precision check. Price lists do not establish transaction-level profit.

In [7]:
common = ['MRP Old', 'Final MRP Old', 'Ajio MRP', 'Amazon MRP', 'Amazon FBA MRP', 'Flipkart MRP', 'Limeroad MRP', 'Myntra MRP', 'Paytm MRP', 'Snapdeal MRP']
statuses = Counter()
for source, tp in (('price_may', ['TP']), ('price_march', ['TP 1', 'TP 2'])):
    for column in tp + common:
        for value in frames[source][column]:
            if not value.strip():
                statuses['blank'] += 1
            else:
                v2, v6 = decimal_value(value, 2, 20), decimal_value(value, 6, 24)
                statuses['parsed' if v2 is not None and v6 is not None and v2 == v6 else 'unparsed'] += 1
print('Price observations:', sum(statuses.values()))
for status in ('parsed', 'blank', 'unparsed'):
    print(status.title() + ' values:', statuses[status])

Price observations: 30590
Parsed values: 29844
Blank values: 0
Unparsed values: 746


## Interpretation

The expense and warehouse comparison files are retained as source observations; their transaction linkage and financial meaning are not established. Inventory is an undated snapshot. Review the [source assessment](../docs/source-assessment.md), [lakehouse models](../lakehouse/models/), and [tests](../lakehouse/tests/) for decisions and limitations. Keep raw data and unreviewed notebook outputs out of Git.